# Bike Count Estimation — Münster Hourly Time-Series Challenge

**Objective:** Predict hourly bike counts using temporal, weather, and lag features across two forecast horizons:
- **Horizon 1:** Next hour ($t+1$)
- **Horizon 2:** Next 24 hours ($t+1$ to $t+24$)

**Metric:** Mean Squared Error (MSE)

**Models:** Linear (Ridge), Tree-Based (LightGBM), Neural Network (PyTorch MLP/LSTM)

## 1. Environment Setup & Imports

In [1]:
import numpy as np
import pandas as pd
import re
import warnings
from datetime import date, timedelta
from pathlib import Path

# Scikit-learn
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from sklearn.multioutput import MultiOutputRegressor

# # Tree-based
import lightgbm as lgb

# # PyTorch
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch device: {DEVICE}")

PyTorch device: cpu


## 2. Data Loading & Initial Exploration

In [2]:
# --- CONFIGURATION ---
# Update this path to point to your training CSV/Excel file
TRAIN_DATA_PATH = Path("challenge_public_dataset.xlsx")

# Load data
if TRAIN_DATA_PATH.suffix == ".xlsx":
    df_raw = pd.read_excel(TRAIN_DATA_PATH)
else:
    df_raw = pd.read_csv(TRAIN_DATA_PATH)

print(f"Dataset shape: {df_raw.shape}")
print(f"Columns: {df_raw.columns.tolist()}")
df_raw.head(10)

Dataset shape: (8760, 10)
Columns: ['Month', 'Day', 'Hour', 'Weekday', 'Weather', 'Temperature (°C)', 'Humidity (%)', 'Rain (mm)', 'Wind (km/h)', 'BikeCount']


,Month,Day,Hour,Weekday,Weather,Temperature (°C),Humidity (%),Rain (mm),Wind (km/h),BikeCount
0,1,1,0,6,Sunny,14.0,61.0,0.0,34.0,73.0
1,1,1,1,6,Sunny,14.0,59.0,0.0,34.0,193.0
2,1,1,2,6,Partly Cloudy,14.0,57.0,0.0,33.0,240.0
3,1,1,3,6,Partly Cloudy,14.0,55.0,0.0,33.0,279.0
4,1,1,4,6,Partly Cloudy,14.0,56.0,0.0,33.0,194.0
5,1,1,5,6,Partly Cloudy,14.0,57.0,0.0,33.0,112.0
6,1,1,6,6,Partly Cloudy,14.0,58.0,0.0,33.0,45.0
7,1,1,7,6,Partly Cloudy,14.0,61.0,0.0,31.0,21.0
8,1,1,8,6,Partly Cloudy,13.0,64.0,0.0,29.0,26.0
9,1,1,9,6,Partly Cloudy,13.0,66.0,0.0,26.0,24.0


In [3]:
df_raw.info()
print("\n--- Descriptive Statistics ---")
df_raw.describe()

<class 'pandas.DataFrame'>
RangeIndex: 8760 entries, 0 to 8759
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Month             8760 non-null   int64  
 1   Day               8760 non-null   int64  
 2   Hour              8760 non-null   int64  
 3   Weekday           8760 non-null   int64  
 4   Weather           8760 non-null   str    
 5   Temperature (°C)  8759 non-null   float64
 6   Humidity (%)      8759 non-null   float64
 7   Rain (mm)         8759 non-null   float64
 8   Wind (km/h)       8759 non-null   float64
 9   BikeCount         8759 non-null   float64
dtypes: float64(5), int64(4), str(1)
memory usage: 684.5 KB

--- Descriptive Statistics ---


,Month,Day,Hour,Weekday,Temperature (°C),Humidity (%),Rain (mm),Wind (km/h),BikeCount
count,8760.000000,8760.000000,8760.000000,8760.000000,8759.000000,8759.000000,8759.000000,8759.000000,8759.000000
mean,6.526027,15.720548,11.500114,3.008219,11.302774,77.102980,0.111622,15.889143,452.342048
std,3.448048,8.796749,6.922433,2.003519,7.138446,15.067433,0.462491,8.505827,363.804893
min,1.000000,1.000000,0.000000,0.000000,-7.000000,25.000000,0.000000,0.000000,0.000000
25%,4.000000,8.000000,5.750000,1.000000,6.000000,68.000000,0.000000,9.000000,119.000000
50%,7.000000,16.000000,11.500000,3.000000,11.000000,80.000000,0.000000,14.000000,380.000000
75%,10.000000,23.000000,17.250000,5.000000,16.000000,89.000000,0.000000,21.500000,730.000000
max,12.000000,31.000000,23.000000,6.000000,36.000000,100.000000,15.800000,48.000000,1789.000000


In [4]:
# Inspect unique values in potentially mixed-text columns
print("Unique 'Weekday' values:")
print(df_raw["Weekday"].unique())
print(f"\nUnique 'Weather' values:")
print(df_raw["Weather"].unique())

Unique 'Weekday' values:
[6 0 1 2 3 4 5]

Unique 'Weather' values:
<StringArray>
[                                        'Sunny',
                                 'Partly Cloudy',
                                      'Overcast',
                               'Occasional Rain',
                                       'Drizzle',
                                    'Light Rain',
                                  'Light Shower',
                                     'Light Fog',
                                        'Cloudy',
    'Occasional Thunderstorms and Precipitation',
                         'Occasional Light Rain',
                            'Occasional Drizzle',
 'Moderate to Heavy Snowfall with Thunderstorms',
                             'Moderate Snowfall',
                                    'Snowdrifts',
                    'Moderate to Heavy Snowfall',
                                     'Snowstorm',
                                'Heavy Snowfall',
                   

## 3. Advanced Feature Engineering Pipeline

This section encapsulates all transformations into reproducible functions that can be applied to unseen test data without data leakage.

**Key transformations:**
1. Parse mixed Weekday/Weather strings into clean categorical columns
2. Cyclical sine/cosine encoding for temporal features (Hour, Day, Month)
3. Lag features ($t-1$, $t-2$, $t-24$) aligned to prevent leakage
4. Rolling window statistics (mean, std) over past windows

In [5]:
def parse_weekday_column(series: pd.Series) -> pd.DataFrame:
    """
    Parse the mixed Weekday column.
    Expected patterns: '6 Sonnig', '6 Leicht bewölkt', '6 Bedeckt', etc.
    Extracts the numeric weekday and the text portion (if present).
    """
    weekday_num = []
    weekday_text = []
    
    for val in series.astype(str):
        # Try to extract leading number and trailing text
        match = re.match(r"^(\d+)\s*(.*)", val.strip())
        if match:
            weekday_num.append(int(match.group(1)))
            text = match.group(2).strip()
            weekday_text.append(text if text else "Unknown")
        else:
            # Fallback: try to use the entire value as categorical
            weekday_num.append(-1)
            weekday_text.append(val.strip())
    
    return pd.DataFrame({
        "weekday_num": weekday_num,
        "weekday_text": weekday_text
    })


def cyclical_encode(value: pd.Series, max_val: float) -> pd.DataFrame:
    """
    Encode a periodic feature using sine/cosine transformation.
    This preserves the cyclical nature (e.g., hour 23 is close to hour 0).
    """
    sin_vals = np.sin(2 * np.pi * value / max_val)
    cos_vals = np.cos(2 * np.pi * value / max_val)
    return sin_vals, cos_vals


def add_lag_features(df: pd.DataFrame, target_col: str = "BikeCount",
                     lags: list = None) -> pd.DataFrame:
    """
    Add lag features for the target variable.
    IMPORTANT: These are strictly backward-looking to prevent data leakage.
    """
    if lags is None:
        lags = [1, 2, 3, 6, 12, 24, 48]
    
    for lag in lags:
        df[f"lag_{lag}"] = df[target_col].shift(lag)
    
    return df


def add_rolling_features(df: pd.DataFrame, target_col: str = "BikeCount",
                         windows: list = None) -> pd.DataFrame:
    """
    Add rolling mean and std over past windows.
    Uses shift(1) to ensure we only look at past data (no current value).
    """
    if windows is None:
        windows = [3, 6, 12, 24]
    
    for w in windows:
        # shift(1) ensures we don't include the current timestep
        rolled = df[target_col].shift(1).rolling(window=w, min_periods=1)
        df[f"rolling_mean_{w}"] = rolled.mean()
        df[f"rolling_std_{w}"] = rolled.std().fillna(0)
    
    return df


# --- Weather severity bucketing ---
# The raw Weather column has 35 categories with very long tails (e.g. "Ice Fog"
# appears <30 times in a year of data). One-hot encoding wastes dimensions and
# starves rare categories of signal. Bucket into ordinal severity + binary
# precipitation/snow flags. Buckets are based on cycling-relevant impact,
# not meteorological taxonomy.
WEATHER_SEVERITY = {
    # 0 = clear / great cycling weather
    "Sunny": 0,
    "Partly Cloudy": 0,
    # 1 = cloudy / no precipitation
    "Cloudy": 1,
    "Overcast": 1,
    "Fog": 1,
    "Light Fog": 1,
    "Ice Fog": 1,
    # 2 = light precipitation
    "Drizzle": 2,
    "Occasional Drizzle": 2,
    "Light Rain": 2,
    "Occasional Light Rain": 2,
    "Light Shower": 2,
    "Occasional Rain": 2,
    "Light Snowfall": 2,
    "Occasional Light Snowfall": 2,
    "Light Snow Showers": 2,
    "Occasional Snowfall": 2,
    "Light Ice Rain": 2,
    "Occasional Ice Rain": 2,
    # 3 = moderate precipitation
    "Moderate Rainfall": 3,
    "Partially Moderate Rainfall": 3,
    "Moderate Snowfall": 3,
    "Occasional Moderate Snowfall": 3,
    "Occasional Drizzle with Thunderstorms": 3,
    # 4 = heavy / dangerous
    "Heavy Rainfall": 4,
    "Partially Heavy Rainfall": 4,
    "Moderate to Heavy Shower": 4,
    "Moderate to Heavy Rain with Thunderstorms": 4,
    "Heavy Snowfall": 4,
    "Moderate to Heavy Snowfall": 4,
    "Moderate to Heavy Snowfall with Thunderstorms": 4,
    "Snowstorm": 4,
    "Snowdrifts": 4,
    "Occasional Thunderstorms and Precipitation": 4,
}

_PRECIP_KEYWORDS = ("rain", "drizzle", "shower", "snow", "thunderstorm", "ice")
_SNOW_KEYWORDS = ("snow", "snowstorm", "snowdrift")


def _weather_severity(label: str) -> int:
    """Default to bucket 1 (cloudy/unknown) for unseen labels."""
    return WEATHER_SEVERITY.get(label, 1)


def _is_precipitation(label: str) -> int:
    s = label.lower()
    return int(any(k in s for k in _PRECIP_KEYWORDS))


def _is_snow(label: str) -> int:
    s = label.lower()
    return int(any(k in s for k in _SNOW_KEYWORDS))


def add_weather_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Replace the high-cardinality Weather column with three numeric features:
      - weather_severity: 0..4 (ordinal)
      - is_precipitation: binary
      - is_snow: binary
    "Weather Condition Null" is mapped to severity 1 + no precipitation.
    """
    df = df.copy()
    weather = df["Weather"].astype(str)
    is_null = weather.str.contains("Null", case=False, na=False)
    weather = weather.where(~is_null, other="Cloudy")
    df["weather_severity"] = weather.map(_weather_severity).astype(np.int64)
    df["is_precipitation"] = weather.map(_is_precipitation).astype(np.int64)
    df["is_snow"] = weather.map(_is_snow).astype(np.int64)
    return df


# --- Sunrise / sunset (closed-form, no external data) ---
# Münster, NRW ≈ 51.96°N, 7.63°E. Daylight is the strongest year-round driver
# of bike traffic at extreme hours: a 6 AM hour in mid-June is full daylight,
# in mid-December it's pitch dark. The cyclical hour features can't express
# this because they treat the hour identically across seasons.
LAT_DEG = 51.96
LON_DEG = 7.63
TZ_OFFSET_HOURS = 1.0  # CET. The model only needs a *consistent* daylight
                       # signal, not the politically-correct DST one.


def _solar_event_hours(month: int, day: int, year: int = 2001) -> tuple[float, float]:
    """
    Sunrise and sunset for the given month/day at Münster, as fractional
    local hours (e.g. 6.42 = 06:25). Year defaults to 2001 because civic
    sunrise varies by <2 minutes year-to-year.

    Uses the NOAA solar position formulas (simplified, pure arithmetic).
    """
    doy = (date(year, month, day) - date(year, 1, 1)).days + 1
    gamma = 2 * np.pi / 365.0 * (doy - 1 + 0.5)

    eq_time = 229.18 * (
        0.000075
        + 0.001868 * np.cos(gamma)
        - 0.032077 * np.sin(gamma)
        - 0.014615 * np.cos(2 * gamma)
        - 0.040849 * np.sin(2 * gamma)
    )

    decl = (
        0.006918
        - 0.399912 * np.cos(gamma)
        + 0.070257 * np.sin(gamma)
        - 0.006758 * np.cos(2 * gamma)
        + 0.000907 * np.sin(2 * gamma)
        - 0.002697 * np.cos(3 * gamma)
        + 0.00148 * np.sin(3 * gamma)
    )

    lat_rad = np.radians(LAT_DEG)
    cos_ha = (np.cos(np.radians(90.833)) - np.sin(lat_rad) * np.sin(decl)) \
             / (np.cos(lat_rad) * np.cos(decl))
    cos_ha = np.clip(cos_ha, -1.0, 1.0)
    ha_deg = np.degrees(np.arccos(cos_ha))

    noon_min = 720.0 - 4.0 * LON_DEG - eq_time + 60.0 * TZ_OFFSET_HOURS
    sunrise_min = noon_min - 4.0 * ha_deg
    sunset_min = noon_min + 4.0 * ha_deg
    return sunrise_min / 60.0, sunset_min / 60.0


def add_daylight_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add `hours_since_sunrise` and `hours_until_sunset` (signed, in local hours).
    Negative `hours_since_sunrise` -> still dark; negative `hours_until_sunset`
    -> sun has already set.
    """
    df = df.copy()
    unique_dates = df[["Month", "Day"]].drop_duplicates()
    solar = {}
    for _, row in unique_dates.iterrows():
        m, d = int(row["Month"]), int(row["Day"])
        solar[(m, d)] = _solar_event_hours(m, d)

    sunrise = df.apply(lambda r: solar[(int(r["Month"]), int(r["Day"]))][0], axis=1)
    sunset = df.apply(lambda r: solar[(int(r["Month"]), int(r["Day"]))][1], axis=1)
    hour_f = df["Hour"].astype(float)
    df["hours_since_sunrise"] = hour_f - sunrise
    df["hours_until_sunset"] = sunset - hour_f
    return df


def build_features(df: pd.DataFrame, is_training: bool = True,
                   target_col: str = "BikeCount",
                   year_override: int | None = None) -> pd.DataFrame:
    """
    Master feature engineering function.
    Applies all transformations in sequence. Can be used for both training and inference.

    Parameters:
        df: Raw dataframe with original columns
        is_training: If True, target column is expected to be present
        target_col: Name of the target variable
        year_override: If provided, skip year inference and use this year for the
            holiday-feature lookup. Useful when the challenge organisers state the
            year of the unseen test data.

    Returns:
        Fully engineered DataFrame
    """
    df = df.copy()
    
    # --- 1. Parse mixed-text Weekday column ---
    weekday_parsed = parse_weekday_column(df["Weekday"])
    df["weekday_num"] = weekday_parsed["weekday_num"]
    df["weekday_text"] = weekday_parsed["weekday_text"]
    df.drop(columns=["Weekday"], inplace=True)

    # --- 1b. German public holiday features (NRW) ---
    if year_override is not None:
        year = year_override
    else:
        df_for_year = df[["Month", "Day"]].copy()
        df_for_year["Weekday"] = df["weekday_num"]
        year = infer_year(df_for_year)
    df = add_holiday_features(df, year=year)

    # --- 1c. Weather severity bucketing (replaces 35-cat one-hot) ---
    df = add_weather_features(df)

    # --- 1d. Daylight features (closed-form astronomy) ---
    df = add_daylight_features(df)

    # --- 2. Cyclical temporal encoding ---
    df["hour_sin"], df["hour_cos"] = cyclical_encode(df["Hour"], 24)
    df["day_sin"], df["day_cos"] = cyclical_encode(df["Day"], 31)
    df["month_sin"], df["month_cos"] = cyclical_encode(df["Month"], 12)
    df["weekday_sin"], df["weekday_cos"] = cyclical_encode(df["weekday_num"], 7)
    
    # --- 3. Binary indicators ---
    df["is_weekend"] = (df["weekday_num"] >= 5).astype(int)
    df["is_rush_hour"] = df["Hour"].isin([7, 8, 9, 16, 17, 18]).astype(int)
    df["is_night"] = df["Hour"].isin(list(range(0, 6))).astype(int)
    
    # --- 4. Lag features (only if target exists) ---
    if target_col in df.columns:
        df = add_lag_features(df, target_col)
        df = add_rolling_features(df, target_col)
    
    # --- 5. Interaction features ---
    df["temp_humidity"] = df["Temperature (°C)"] * df["Humidity (%)"]
    df["wind_rain"] = df["Wind (km/h)"] * df["Rain (mm)"]
    
    return df


print("Feature engineering functions defined.")

Feature engineering functions defined.


### German Public Holiday Features (NRW)

Adds `is_holiday` and `is_bridge_day` flags so the model can distinguish German public holidays (which behave like weekend days for bike traffic) from regular weekdays.

- **Year inference:** The dataset has no year column. `infer_year()` deterministically picks the year whose Gregorian calendar matches every row's `(Month, Day) → Weekday`. If the test data is from an ambiguous year (e.g. two non-leap years share the same Jan-1 weekday), pass `year_override=` to skip inference.
- **NRW holiday set:** Computed from `year` — no static lookup. Easter Sunday is derived via Gauss's algorithm; the other moving holidays (Karfreitag, Ostermontag, Christi Himmelfahrt, Pfingstmontag, Fronleichnam) are offsets from Easter. Fixed: Neujahr, Tag der Arbeit, Tag der Deutschen Einheit, Allerheiligen, 1. & 2. Weihnachtstag.
- **Bridge day:** A non-holiday weekday wedged between a holiday and a weekend (e.g. Friday after a Thursday holiday).

In [6]:
def _easter_sunday(year: int) -> date:
    """Gauss's algorithm — Easter Sunday for a given Gregorian year."""
    a = year % 19
    b = year // 100
    c = year % 100
    d = b // 4
    e = b % 4
    f = (b + 8) // 25
    g = (b - f + 1) // 3
    h = (19 * a + b - d - g + 15) % 30
    i = c // 4
    k = c % 4
    l = (32 + 2 * e + 2 * i - h - k) % 7
    m = (a + 11 * h + 22 * l) // 451
    month = (h + l - 7 * m + 114) // 31
    day = ((h + l - 7 * m + 114) % 31) + 1
    return date(year, month, day)


def nrw_holidays(year: int) -> dict[date, str]:
    """Return all NRW public holidays for `year` as {date: name}."""
    easter = _easter_sunday(year)
    holidays = {
        date(year, 1, 1):   "Neujahr",
        date(year, 5, 1):   "Tag der Arbeit",
        date(year, 10, 3):  "Tag der Deutschen Einheit",
        date(year, 11, 1):  "Allerheiligen",
        date(year, 12, 25): "1. Weihnachtstag",
        date(year, 12, 26): "2. Weihnachtstag",
        easter + timedelta(days=-2): "Karfreitag",
        easter + timedelta(days=1):  "Ostermontag",
        easter + timedelta(days=39): "Christi Himmelfahrt",
        easter + timedelta(days=50): "Pfingstmontag",
        easter + timedelta(days=60): "Fronleichnam",
    }
    return holidays


def infer_year(df: pd.DataFrame, candidate_range: tuple = (2010, 2030)) -> int:
    """
    Infer the year of the dataset by matching (Month, Day) -> Weekday alignment.
    The dataset uses Python's convention: Monday=0 ... Sunday=6.

    If multiple years match, returns the most recent one and warns.
    """
    # Sample one row per (Month, Day) — fastest unique check
    sample = df[["Month", "Day", "Weekday"]].drop_duplicates(subset=["Month", "Day"])

    matches = []
    for yr in range(candidate_range[0], candidate_range[1] + 1):
        ok = True
        for _, row in sample.iterrows():
            try:
                d = date(yr, int(row["Month"]), int(row["Day"]))
            except ValueError:
                # e.g. Feb 29 in a non-leap year — skip this candidate
                ok = False
                break
            if d.weekday() != int(row["Weekday"]):
                ok = False
                break
        if ok:
            matches.append(yr)

    if not matches:
        raise ValueError(
            f"No year in {candidate_range} matches the (Month, Day) -> Weekday alignment. "
            "Check that Weekday uses Mon=0..Sun=6."
        )
    if len(matches) > 1:
        warnings.warn(
            f"Multiple candidate years match the calendar: {matches}. "
            f"Using most recent ({matches[-1]}). Pass year_override= to disambiguate."
        )
    return matches[-1]


def add_holiday_features(df: pd.DataFrame, year: int) -> pd.DataFrame:
    """
    Add `is_holiday` and `is_bridge_day` columns based on NRW public holidays for `year`.
    Pure function — does not mutate the input.
    """
    df = df.copy()
    holidays = nrw_holidays(year)
    holiday_dates = set(holidays.keys())

    # Build (date) -> is_holiday lookup for the full year, then derive bridge days
    jan1 = date(year, 1, 1)
    days_in_year = 366 if (year % 4 == 0 and (year % 100 != 0 or year % 400 == 0)) else 365
    all_days = [jan1 + timedelta(days=i) for i in range(days_in_year)]
    holiday_flag = {d: (d in holiday_dates) for d in all_days}

    # Bridge day: a non-holiday Mon/Tue/Wed/Thu/Fri whose adjacent weekday is a holiday,
    # AND that adjacency closes a gap to the weekend.
    bridge_flag = {}
    for d in all_days:
        wd = d.weekday()  # Mon=0..Sun=6
        if holiday_flag[d] or wd >= 5:
            bridge_flag[d] = False
            continue
        prev_day = d - timedelta(days=1)
        next_day = d + timedelta(days=1)
        # Friday after Thursday-holiday (Sat-Sun weekend follows)
        if wd == 4 and holiday_flag.get(prev_day, False):
            bridge_flag[d] = True
        # Monday before Tuesday-holiday (Sat-Sun weekend precedes)
        elif wd == 0 and holiday_flag.get(next_day, False):
            bridge_flag[d] = True
        else:
            bridge_flag[d] = False

    # Vectorised lookup over the dataframe
    row_dates = [date(year, int(m), int(d)) for m, d in zip(df["Month"], df["Day"])]
    df["is_holiday"] = np.array([1 if holiday_flag[d] else 0 for d in row_dates], dtype=np.int64)
    df["is_bridge_day"] = np.array([1 if bridge_flag[d] else 0 for d in row_dates], dtype=np.int64)
    return df


print("Holiday-feature functions defined.")

Holiday-feature functions defined.


In [7]:
# Apply feature engineering to training data
df_feat = build_features(df_raw, is_training=True)

print(f"Engineered dataset shape: {df_feat.shape}")
print(f"\nNew columns: {df_feat.columns.tolist()}")
df_feat.head()

Engineered dataset shape: (8760, 46)

New columns: ['Month', 'Day', 'Hour', 'Weather', 'Temperature (°C)', 'Humidity (%)', 'Rain (mm)', 'Wind (km/h)', 'BikeCount', 'weekday_num', 'weekday_text', 'is_holiday', 'is_bridge_day', 'weather_severity', 'is_precipitation', 'is_snow', 'hours_since_sunrise', 'hours_until_sunset', 'hour_sin', 'hour_cos', 'day_sin', 'day_cos', 'month_sin', 'month_cos', 'weekday_sin', 'weekday_cos', 'is_weekend', 'is_rush_hour', 'is_night', 'lag_1', 'lag_2', 'lag_3', 'lag_6', 'lag_12', 'lag_24', 'lag_48', 'rolling_mean_3', 'rolling_std_3', 'rolling_mean_6', 'rolling_std_6', 'rolling_mean_12', 'rolling_std_12', 'rolling_mean_24', 'rolling_std_24', 'temp_humidity', 'wind_rain']


,Month,Day,Hour,Weather,Temperature (°C),Humidity (%),Rain (mm),Wind (km/h),BikeCount,weekday_num,weekday_text,is_holiday,is_bridge_day,weather_severity,is_precipitation,is_snow,hours_since_sunrise,hours_until_sunset,hour_sin,hour_cos,day_sin,day_cos,month_sin,month_cos,weekday_sin,weekday_cos,is_weekend,is_rush_hour,is_night,lag_1,lag_2,lag_3,lag_6,lag_12,lag_24,lag_48,rolling_mean_3,rolling_std_3,rolling_mean_6,rolling_std_6,rolling_mean_12,rolling_std_12,rolling_mean_24,rolling_std_24,temp_humidity,wind_rain
0,1,1,0,Sunny,14.0,61.0,0.0,34.0,73.0,6,Unknown,1,0,0,0,0,-8.620727,16.46621,0.000000,1.000000,0.201299,0.97953,0.5,0.866025,-0.781831,0.62349,1,0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,0.000000,NaN,0.000000,NaN,0.000000,854.0,0.0
1,1,1,1,Sunny,14.0,59.0,0.0,34.0,193.0,6,Unknown,1,0,0,0,0,-7.620727,15.46621,0.258819,0.965926,0.201299,0.97953,0.5,0.866025,-0.781831,0.62349,1,0,1,73.0,NaN,NaN,NaN,NaN,NaN,NaN,73.000000,0.000000,73.000000,0.000000,73.000000,0.000000,73.000000,0.000000,826.0,0.0
2,1,1,2,Partly Cloudy,14.0,57.0,0.0,33.0,240.0,6,Unknown,1,0,0,0,0,-6.620727,14.46621,0.500000,0.866025,0.201299,0.97953,0.5,0.866025,-0.781831,0.62349,1,0,1,193.0,73.0,NaN,NaN,NaN,NaN,NaN,133.000000,84.852814,133.000000,84.852814,133.000000,84.852814,133.000000,84.852814,798.0,0.0
3,1,1,3,Partly Cloudy,14.0,55.0,0.0,33.0,279.0,6,Unknown,1,0,0,0,0,-5.620727,13.46621,0.707107,0.707107,0.201299,0.97953,0.5,0.866025,-0.781831,0.62349,1,0,1,240.0,193.0,73.0,NaN,NaN,NaN,NaN,168.666667,86.118136,168.666667,86.118136,168.666667,86.118136,168.666667,86.118136,770.0,0.0
4,1,1,4,Partly Cloudy,14.0,56.0,0.0,33.0,194.0,6,Unknown,1,0,0,0,0,-4.620727,12.46621,0.866025,0.500000,0.201299,0.97953,0.5,0.866025,-0.781831,0.62349,1,0,1,279.0,240.0,193.0,NaN,NaN,NaN,NaN,237.333333,43.061971,196.250000,89.373281,196.250000,89.373281,196.250000,89.373281,784.0,0.0


In [8]:
# --- Holiday-feature sanity check ---
# Confirms the inferred year is plausible by inspecting bike traffic on known holidays.
# Expectation: holidays should look more like weekends than like a regular workday,
# and 1. Mai (warm, leisure-driven) should differ clearly from a normal Wednesday.

_year = infer_year(df_raw[["Month", "Day", "Weekday"]])
print(f"Inferred year: {_year}")
print(f"\nNRW public holidays for {_year}:")
for d, name in sorted(nrw_holidays(_year).items()):
    print(f"  {d.isoformat()} ({['Mon','Tue','Wed','Thu','Fri','Sat','Sun'][d.weekday()]:<3}) — {name}")

# Aggregate daily bike counts and compare holidays to same-weekday non-holidays
_daily = (df_feat.groupby(["Month", "Day"], as_index=False)
                  .agg(BikeCount=("BikeCount", "sum"),
                       is_holiday=("is_holiday", "max"),
                       is_bridge_day=("is_bridge_day", "max"),
                       weekday_num=("weekday_num", "first")))

print("\n--- Daily total BikeCount: holidays vs. non-holidays ---")
print(_daily.groupby("is_holiday")["BikeCount"].agg(["count", "mean", "median"]).round(0))

print("\n--- Same-weekday comparison (only weekdays on which a holiday fell) ---")
holiday_weekdays = _daily.loc[_daily["is_holiday"] == 1, "weekday_num"].unique()
for wd in sorted(holiday_weekdays):
    wd_name = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"][wd]
    sub = _daily[_daily["weekday_num"] == wd]
    hol = sub.loc[sub["is_holiday"] == 1, "BikeCount"]
    non = sub.loc[sub["is_holiday"] == 0, "BikeCount"]
    if len(hol) and len(non):
        print(f"  {wd_name}: holiday mean={hol.mean():>6.0f} (n={len(hol)})  |  non-holiday mean={non.mean():>6.0f} (n={len(non)})")

print("\n--- Spot checks ---")
def _show(month, day, label):
    row = _daily[(_daily["Month"] == month) & (_daily["Day"] == day)]
    if not row.empty:
        bc = row["BikeCount"].iloc[0]
        hol = bool(row["is_holiday"].iloc[0])
        brg = bool(row["is_bridge_day"].iloc[0])
        wd = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"][int(row["weekday_num"].iloc[0])]
        print(f"  {month:02d}-{day:02d} ({wd}): daily total = {bc:>6.0f}  |  holiday={hol}  bridge={brg}  |  {label}")

_show(5, 1,  "Tag der Arbeit (expect: low commute, leisure-shifted)")
_show(12, 25, "1. Weihnachtstag (expect: very low)")
_show(12, 26, "2. Weihnachtstag (expect: very low)")
_show(10, 3, "Tag der Deutschen Einheit")
easter_mon = _easter_sunday(_year) + timedelta(days=1)
_show(easter_mon.month, easter_mon.day, "Ostermontag")
himmelfahrt = _easter_sunday(_year) + timedelta(days=39)
_show(himmelfahrt.month, himmelfahrt.day, "Christi Himmelfahrt")

Inferred year: 2023

NRW public holidays for 2023:
  2023-01-01 (Sun) — Neujahr
  2023-04-07 (Fri) — Karfreitag
  2023-04-10 (Mon) — Ostermontag
  2023-05-01 (Mon) — Tag der Arbeit
  2023-05-18 (Thu) — Christi Himmelfahrt
  2023-05-29 (Mon) — Pfingstmontag
  2023-06-08 (Thu) — Fronleichnam
  2023-10-03 (Tue) — Tag der Deutschen Einheit
  2023-11-01 (Wed) — Allerheiligen
  2023-12-25 (Mon) — 1. Weihnachtstag
  2023-12-26 (Tue) — 2. Weihnachtstag

--- Daily total BikeCount: holidays vs. non-holidays ---
            count     mean   median
is_holiday                         
0             354  11019.0  11314.0
1              11   5591.0   6166.0

--- Same-weekday comparison (only weekdays on which a holiday fell) ---
  Mon: holiday mean=  4861 (n=4)  |  non-holiday mean= 12603 (n=48)
  Tue: holiday mean=  5547 (n=2)  |  non-holiday mean= 13623 (n=50)
  Wed: holiday mean=  7795 (n=1)  |  non-holiday mean= 13683 (n=51)
  Thu: holiday mean=  8666 (n=2)  |  non-holiday mean= 12620 (n=50)
  Fr

In [9]:
# --- Prepare final feature matrix ---
# Define which columns are numeric vs. categorical for the preprocessor

TARGET_COL = "BikeCount"

# Categorical columns to one-hot encode.
# Weather is no longer here — it's been bucketed into `weather_severity`,
# `is_precipitation`, `is_snow` (all numeric) by add_weather_features().
CAT_COLS = ["weekday_text"]

# Columns to drop. Raw temporal columns are already encoded cyclically;
# raw Weather has been replaced by numeric severity features above.
DROP_COLS = ["Month", "Day", "Hour", "weekday_num", "Weather", TARGET_COL]

# Numeric feature columns (everything else)
NUM_COLS = [c for c in df_feat.columns if c not in CAT_COLS + DROP_COLS]

print(f"Numeric features ({len(NUM_COLS)}): {NUM_COLS}")
print(f"Categorical features ({len(CAT_COLS)}): {CAT_COLS}")

Numeric features (39): ['Temperature (°C)', 'Humidity (%)', 'Rain (mm)', 'Wind (km/h)', 'is_holiday', 'is_bridge_day', 'weather_severity', 'is_precipitation', 'is_snow', 'hours_since_sunrise', 'hours_until_sunset', 'hour_sin', 'hour_cos', 'day_sin', 'day_cos', 'month_sin', 'month_cos', 'weekday_sin', 'weekday_cos', 'is_weekend', 'is_rush_hour', 'is_night', 'lag_1', 'lag_2', 'lag_3', 'lag_6', 'lag_12', 'lag_24', 'lag_48', 'rolling_mean_3', 'rolling_std_3', 'rolling_mean_6', 'rolling_std_6', 'rolling_mean_12', 'rolling_std_12', 'rolling_mean_24', 'rolling_std_24', 'temp_humidity', 'wind_rain']
Categorical features (1): ['weekday_text']


## 4. Data Split (Time-Series Ground Rules)

**Critical:** We use a strict temporal split — no shuffling. The validation set is always chronologically after the training set to simulate real-world forecasting.

In [10]:
# --- Create multi-horizon targets ---
# Horizon 1: predict t+1
df_feat["target_h1"] = df_feat[TARGET_COL].shift(-1)

# Horizon 2: predict t+1 to t+24
for h in range(1, 25):
    df_feat[f"target_h{h}"] = df_feat[TARGET_COL].shift(-h)

# Identify lag/rolling columns and target columns
lag_cols = [c for c in df_feat.columns if c.startswith("lag_") or c.startswith("rolling_")]
target_h24_cols_all = [f"target_h{h}" for h in range(1, 25)]

# Drop rows where ANY feature or target column contains NaN
# This handles: lag boundaries, target boundaries, AND missing values in original data
df_clean = df_feat.dropna(subset=NUM_COLS + CAT_COLS + lag_cols + target_h24_cols_all).copy().reset_index(drop=True)
print(f"Usable samples after removing NaN rows: {len(df_clean)}")

# Verify no NaNs remain in feature columns
assert df_clean[NUM_COLS + lag_cols].isna().sum().sum() == 0, "Features still contain NaNs!"

Usable samples after removing NaN rows: 8656


In [11]:
# --- Temporal Train/Validation Split ---
# Use last ~20% of data as validation (strictly chronological)
SPLIT_RATIO = 0.8
split_idx = int(len(df_clean) * SPLIT_RATIO)

df_train = df_clean.iloc[:split_idx].copy()
df_val = df_clean.iloc[split_idx:].copy()

print(f"Training samples: {len(df_train)}")
print(f"Validation samples: {len(df_val)}")
print(f"Train period: rows 0–{split_idx-1}")
print(f"Val period:   rows {split_idx}–{len(df_clean)-1}")

Training samples: 6924
Validation samples: 1732
Train period: rows 0–6923
Val period:   rows 6924–8655


In [12]:
# --- Build sklearn ColumnTransformer for preprocessing ---
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), NUM_COLS),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT_COLS),
    ],
    remainder="drop"
)

# Fit on training data only
X_train = preprocessor.fit_transform(df_train)
X_val = preprocessor.transform(df_val)

# --- Targets (raw counts) ---
# These are what the grader's MSE is computed against.
y_train_h1 = df_train["target_h1"].values
y_val_h1 = df_val["target_h1"].values

target_h24_cols = [f"target_h{h}" for h in range(1, 25)]
y_train_h24 = df_train[target_h24_cols].values
y_val_h24 = df_val[target_h24_cols].values

# --- Log-space targets ---
# BikeCount is heavily right-skewed (range 0-1789, median 380, mean 452).
# Training on log1p(y) makes the model focus equally on busy and quiet hours,
# prevents negative predictions, and gives LightGBM/MLP a much easier
# distribution to fit. We `expm1` predictions before reporting MSE so the
# evaluation metric stays in raw counts.
y_train_h1_log = np.log1p(y_train_h1)
y_val_h1_log = np.log1p(y_val_h1)
y_train_h24_log = np.log1p(y_train_h24)
y_val_h24_log = np.log1p(y_val_h24)


def back_transform(log_pred: np.ndarray) -> np.ndarray:
    """Inverse of log1p, clipped to non-negative counts."""
    return np.clip(np.expm1(log_pred), 0.0, None)


print(f"X_train shape: {X_train.shape}")
print(f"X_val shape:   {X_val.shape}")
print(f"y_train_h1 shape: {y_train_h1.shape}  (raw counts; log version also available)")
print(f"y_train_h24 shape: {y_train_h24.shape}")
print(f"\nTarget stats (raw):     mean={y_train_h1.mean():.1f}  std={y_train_h1.std():.1f}  max={y_train_h1.max():.0f}")
print(f"Target stats (log1p):   mean={y_train_h1_log.mean():.2f}  std={y_train_h1_log.std():.2f}  max={y_train_h1_log.max():.2f}")

X_train shape: (6924, 40)
X_val shape:   (1732, 40)
y_train_h1 shape: (6924,)  (raw counts; log version also available)
y_train_h24 shape: (6924, 24)

Target stats (raw):     mean=456.9  std=357.2  max=1689
Target stats (log1p):   mean=5.60  std=1.26  max=7.43


## 5. Model Implementations & Training

### 5.1 Linear Model — Ridge Regression

In [13]:
# --- Horizon 1: Single-step Ridge (raw target) ---
# log1p target was tested and made Ridge dramatically worse (47k vs 23k MSE):
# Ridge fits log-space linearly, but the expm1 back-transform amplifies tiny
# log-space errors into huge raw-count errors at the busy end. Stick with raw.
ridge_h1 = Ridge(alpha=1.0)
ridge_h1.fit(X_train, y_train_h1)

pred_ridge_h1 = ridge_h1.predict(X_val)
mse_ridge_h1 = mean_squared_error(y_val_h1, pred_ridge_h1)
print(f"Ridge — Horizon 1 (t+1) MSE: {mse_ridge_h1:.2f}")

Ridge — Horizon 1 (t+1) MSE: 21630.64


In [14]:
# --- Horizon 2: Multi-output Ridge for 24-step ahead (raw target) ---
ridge_h24 = MultiOutputRegressor(Ridge(alpha=1.0))
ridge_h24.fit(X_train, y_train_h24)

pred_ridge_h24 = ridge_h24.predict(X_val)
mse_ridge_h24 = mean_squared_error(y_val_h24, pred_ridge_h24)
print(f"Ridge — Horizon 2 (t+1..t+24) MSE: {mse_ridge_h24:.2f}")

Ridge — Horizon 2 (t+1..t+24) MSE: 38127.69


### 5.2 Tree-Based Model — LightGBM

In [ ]:
# --- Horizon 1: Single-step LightGBM (raw target) ---
# Tested log-target; LightGBM was slightly worse with it (7666 vs 6447 MSE).
# Trees split on order, not magnitude, so they tolerate skewed targets fine —
# and the log-transform removed signal at the high end where the squared-error
# budget concentrates.
lgb_params = {
    "objective": "regression",
    "metric": "mse",
    "learning_rate": 0.05,
    "num_leaves": 63,
    "max_depth": -1,
    "min_child_samples": 20,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 5,
    "n_estimators": 1000,
    "early_stopping_rounds": 50,
    "verbose": -1,
    "random_state": RANDOM_SEED,
}

lgb_h1 = lgb.LGBMRegressor(**lgb_params)
lgb_h1.fit(
    X_train, y_train_h1,
    eval_set=[(X_val, y_val_h1)],
)

pred_lgb_h1 = lgb_h1.predict(X_val)
mse_lgb_h1 = mean_squared_error(y_val_h1, pred_lgb_h1)
print(f"\nLightGBM — Horizon 1 (t+1) MSE: {mse_lgb_h1:.2f}")

In [ ]:
# --- Horizon 2: Train separate LightGBM for each of the 24 steps (raw target) ---
# Direct multi-step strategy: one model per horizon.
lgb_h24_models = []
pred_lgb_h24 = np.zeros((len(X_val), 24))

lgb_params_h24 = lgb_params.copy()
lgb_params_h24["n_estimators"] = 500  # slightly fewer for speed

for h in range(24):
    model = lgb.LGBMRegressor(**lgb_params_h24)
    model.fit(
        X_train, y_train_h24[:, h],
        eval_set=[(X_val, y_val_h24[:, h])],
    )
    lgb_h24_models.append(model)
    pred_lgb_h24[:, h] = model.predict(X_val)

mse_lgb_h24 = mean_squared_error(y_val_h24, pred_lgb_h24)
print(f"LightGBM — Horizon 2 (t+1..t+24) MSE: {mse_lgb_h24:.2f}")

### 5.3 Neural Network — PyTorch MLP

A multi-layer perceptron with residual connections, batch normalization, and dropout for regularization.

In [ ]:
class BikeCountDataset(Dataset):
    """PyTorch Dataset for tabular bike count features."""
    
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


class BikeCountMLP(nn.Module):
    """
    Multi-Layer Perceptron for bike count regression.
    Supports single-output (Horizon 1) or multi-output (Horizon 2).
    """
    
    def __init__(self, input_dim: int, output_dim: int = 1,
                 hidden_dims: list = None, dropout: float = 0.2):
        super().__init__()
        if hidden_dims is None:
            hidden_dims = [256, 128, 64]
        
        layers = []
        prev_dim = input_dim
        
        for h_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, h_dim),
                nn.BatchNorm1d(h_dim),
                nn.ReLU(),
                nn.Dropout(dropout),
            ])
            prev_dim = h_dim
        
        layers.append(nn.Linear(prev_dim, output_dim))
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.network(x)


def train_mlp(model, train_loader, val_X, val_y, epochs=100, lr=1e-3,
              patience=15):
    """
    Training loop with early stopping based on validation MSE.
    """
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=5
    )
    criterion = nn.MSELoss()
    
    val_X_tensor = torch.tensor(val_X, dtype=torch.float32).to(DEVICE)
    val_y_tensor = torch.tensor(val_y, dtype=torch.float32).to(DEVICE)
    if val_y_tensor.ndim == 1:
        val_y_tensor = val_y_tensor.unsqueeze(1)
    
    best_val_mse = float("inf")
    best_state = None
    patience_counter = 0
    
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        
        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)
            if y_batch.ndim == 1:
                y_batch = y_batch.unsqueeze(1)
            
            optimizer.zero_grad()
            preds = model(X_batch)
            loss = criterion(preds, y_batch)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * len(X_batch)
        
        # Validation
        model.eval()
        with torch.no_grad():
            val_preds = model(val_X_tensor)
            val_mse = criterion(val_preds, val_y_tensor).item()
        
        scheduler.step(val_mse)
        
        # Early stopping
        if val_mse < best_val_mse:
            best_val_mse = val_mse
            best_state = model.state_dict().copy()
            patience_counter = 0
        else:
            patience_counter += 1
        
        if patience_counter >= patience:
            print(f"  Early stopping at epoch {epoch+1}")
            break
        
        if (epoch + 1) % 20 == 0:
            print(f"  Epoch {epoch+1}: train_loss={epoch_loss/len(train_loader.dataset):.4f}, val_mse={val_mse:.2f}")
    
    # Restore best weights
    model.load_state_dict(best_state)
    return model, best_val_mse


print("Neural network components defined.")

In [ ]:
# --- Horizon 1: Train MLP for single-step prediction (log-target) ---
BATCH_SIZE = 128
EPOCHS = 150

# Train the MLP in log space — much more stable convergence than fitting
# raw counts that range 0-1800.
train_dataset_h1 = BikeCountDataset(X_train, y_train_h1_log)
train_loader_h1 = DataLoader(train_dataset_h1, batch_size=BATCH_SIZE, shuffle=False)
# Note: shuffle=False to preserve temporal ordering within batches

input_dim = X_train.shape[1]
mlp_h1 = BikeCountMLP(input_dim=input_dim, output_dim=1,
                       hidden_dims=[256, 128, 64]).to(DEVICE)

print(f"MLP Architecture (Horizon 1):\n{mlp_h1}")
print(f"\nTotal parameters: {sum(p.numel() for p in mlp_h1.parameters()):,}")
print("\nTraining...")

mlp_h1, best_mse_h1 = train_mlp(
    mlp_h1, train_loader_h1, X_val, y_val_h1_log,
    epochs=EPOCHS, lr=1e-3, patience=20
)

# Final prediction — back-transform to raw counts before scoring
mlp_h1.eval()
with torch.no_grad():
    pred_mlp_h1_log = mlp_h1(torch.tensor(X_val, dtype=torch.float32).to(DEVICE))
    pred_mlp_h1_log = pred_mlp_h1_log.cpu().numpy().flatten()
pred_mlp_h1 = back_transform(pred_mlp_h1_log)

mse_mlp_h1 = mean_squared_error(y_val_h1, pred_mlp_h1)
print(f"\nMLP — Horizon 1 (t+1) MSE: {mse_mlp_h1:.2f}")

In [ ]:
# --- Horizon 2: Train MLP for 24-step ahead prediction (log-target) ---
train_dataset_h24 = BikeCountDataset(X_train, y_train_h24_log)
train_loader_h24 = DataLoader(train_dataset_h24, batch_size=BATCH_SIZE, shuffle=False)

mlp_h24 = BikeCountMLP(input_dim=input_dim, output_dim=24,
                        hidden_dims=[512, 256, 128]).to(DEVICE)

print(f"MLP Architecture (Horizon 2):\n{mlp_h24}")
print(f"\nTotal parameters: {sum(p.numel() for p in mlp_h24.parameters()):,}")
print("\nTraining...")

mlp_h24, best_mse_h24 = train_mlp(
    mlp_h24, train_loader_h24, X_val, y_val_h24_log,
    epochs=EPOCHS, lr=1e-3, patience=20
)

# Final prediction — back-transform to raw counts before scoring
mlp_h24.eval()
with torch.no_grad():
    pred_mlp_h24_log = mlp_h24(torch.tensor(X_val, dtype=torch.float32).to(DEVICE))
    pred_mlp_h24_log = pred_mlp_h24_log.cpu().numpy()
pred_mlp_h24 = back_transform(pred_mlp_h24_log)

mse_mlp_h24 = mean_squared_error(y_val_h24, pred_mlp_h24)
print(f"\nMLP — Horizon 2 (t+1..t+24) MSE: {mse_mlp_h24:.2f}")

## 6. Comparative Performance Evaluation

MSE comparison across all models and both forecast horizons.

In [ ]:
# --- Results Summary ---
results = pd.DataFrame({
    "Model": ["Ridge Regression", "LightGBM", "MLP (PyTorch)"],
    "Horizon 1 MSE (t+1)": [mse_ridge_h1, mse_lgb_h1, mse_mlp_h1],
    "Horizon 2 MSE (t+1..t+24)": [mse_ridge_h24, mse_lgb_h24, mse_mlp_h24],
})

# Add rank columns
results["H1 Rank"] = results["Horizon 1 MSE (t+1)"].rank().astype(int)
results["H2 Rank"] = results["Horizon 2 MSE (t+1..t+24)"].rank().astype(int)

print("=" * 70)
print("          COMPARATIVE MODEL PERFORMANCE (Validation Set)")
print("=" * 70)
print(results.to_string(index=False))
print("=" * 70)

# Highlight best
best_h1 = results.loc[results["Horizon 1 MSE (t+1)"].idxmin(), "Model"]
best_h24 = results.loc[results["Horizon 2 MSE (t+1..t+24)"].idxmin(), "Model"]
print(f"\n🏆 Best Horizon 1: {best_h1}")
print(f"🏆 Best Horizon 2: {best_h24}")

In [ ]:
# --- Optional: Per-horizon MSE breakdown for 24-step models ---
per_hour_mse = pd.DataFrame({
    "Hour Ahead": range(1, 25),
    "Ridge MSE": [mean_squared_error(y_val_h24[:, h], pred_ridge_h24[:, h]) for h in range(24)],
    "LightGBM MSE": [mean_squared_error(y_val_h24[:, h], pred_lgb_h24[:, h]) for h in range(24)],
    "MLP MSE": [mean_squared_error(y_val_h24[:, h], pred_mlp_h24[:, h]) for h in range(24)],
})

print("\nPer-Hour-Ahead MSE Breakdown:")
print(per_hour_mse.to_string(index=False))

## 7. Production Evaluation Function (Inference Block)

A self-contained function for final evaluation on an unseen test set. Accepts a new data path and a trained model pipeline, applies all preprocessing, generates predictions, and computes MSE.

In [ ]:
def evaluate_final_model(
    new_csv_path: str,
    trained_model_pipeline: dict,
    horizon: int = 1,
    year_override: int | None = None,
) -> dict:
    """
    Production evaluation function for the Bike Count Estimation challenge.

    Ridge and LightGBM are trained on raw BikeCount; the MLP is trained on
    log1p(BikeCount) and its predictions need an `expm1` back-transform. This
    function handles both via the pipeline's `model_type` flag.

    Parameters:
        new_csv_path: Path to the new evaluation dataset (CSV or Excel).
        trained_model_pipeline: Dictionary containing:
            - 'preprocessor': Fitted sklearn ColumnTransformer
            - 'model_h1': Trained model for Horizon 1
            - 'model_h24': Trained model for Horizon 2 (or list of models)
            - 'model_type': One of 'ridge', 'lgb', 'mlp' — controls back-transform
            - 'num_cols': List of numeric feature column names
            - 'cat_cols': List of categorical feature column names
        horizon: 1 for single-step, 24 for multi-step
        year_override: Optional year for the holiday-feature lookup. Pass this when
            the challenge organisers tell you what year the unseen data is from.

    Returns:
        Dictionary with 'predictions' (raw counts), 'mse' (if targets available),
        and metadata.
    """
    path = Path(new_csv_path)
    
    # --- Load data ---
    if path.suffix == ".xlsx":
        df_new = pd.read_excel(path)
    else:
        df_new = pd.read_csv(path)
    
    print(f"Loaded evaluation data: {df_new.shape}")
    
    # --- Apply feature engineering ---
    has_target = "BikeCount" in df_new.columns
    df_new_feat = build_features(df_new, is_training=has_target,
                                  year_override=year_override)
    
    # --- Create targets if available ---
    if has_target:
        if horizon == 1:
            df_new_feat["target_h1"] = df_new_feat["BikeCount"].shift(-1)
        else:
            for h in range(1, 25):
                df_new_feat[f"target_h{h}"] = df_new_feat["BikeCount"].shift(-h)
    
    # --- Remove rows with NaN in any feature or target column ---
    num_cols = trained_model_pipeline["num_cols"]
    cat_cols = trained_model_pipeline["cat_cols"]
    eval_lag_cols = [c for c in df_new_feat.columns if c.startswith("lag_") or c.startswith("rolling_")]
    
    dropna_cols = num_cols + cat_cols + eval_lag_cols
    if has_target:
        if horizon == 1:
            dropna_cols += ["target_h1"]
        else:
            dropna_cols += [f"target_h{h}" for h in range(1, 25)]
    dropna_cols = [c for c in dropna_cols if c in df_new_feat.columns]
    
    df_eval = df_new_feat.dropna(subset=dropna_cols).copy().reset_index(drop=True)
    
    # --- Transform features ---
    pre = trained_model_pipeline["preprocessor"]
    X_eval = pre.transform(df_eval)
    
    # --- Generate predictions ---
    # MLP output lives in log space; Ridge/LGBM live in raw count space.
    model_type = trained_model_pipeline["model_type"]
    is_log_space = (model_type == "mlp")
    
    if horizon == 1:
        model = trained_model_pipeline["model_h1"]
        if model_type == "mlp":
            model.eval()
            with torch.no_grad():
                X_tensor = torch.tensor(X_eval, dtype=torch.float32).to(DEVICE)
                raw_pred = model(X_tensor).cpu().numpy().flatten()
        else:
            raw_pred = model.predict(X_eval)
    else:
        model = trained_model_pipeline["model_h24"]
        if model_type == "mlp":
            model.eval()
            with torch.no_grad():
                X_tensor = torch.tensor(X_eval, dtype=torch.float32).to(DEVICE)
                raw_pred = model(X_tensor).cpu().numpy()
        elif model_type == "lgb":
            raw_pred = np.column_stack([m.predict(X_eval) for m in model])
        else:
            raw_pred = model.predict(X_eval)
    
    if is_log_space:
        predictions = np.clip(np.expm1(raw_pred), 0.0, None)
    else:
        predictions = np.clip(raw_pred, 0.0, None)
    
    # --- Compute MSE on raw counts (the grader's metric) ---
    result = {"predictions": predictions, "n_samples": len(df_eval)}
    
    if has_target:
        if horizon == 1:
            y_true = df_eval["target_h1"].values
        else:
            target_cols = [f"target_h{h}" for h in range(1, 25)]
            y_true = df_eval[target_cols].values
        
        mse = mean_squared_error(y_true, predictions)
        result["mse"] = mse
        print(f"\n{'='*50}")
        print(f"  FINAL EVALUATION MSE (Horizon {horizon}): {mse:.4f}")
        print(f"{'='*50}")
    else:
        print("No target column found — returning predictions only.")
    
    return result


print("evaluate_final_model() defined and ready.")

In [ ]:
# --- Example usage: Package the best model into a pipeline dict ---

# LightGBM pipeline (typically strongest baseline for tabular data)
lgb_pipeline = {
    "preprocessor": preprocessor,
    "model_h1": lgb_h1,
    "model_h24": lgb_h24_models,
    "model_type": "lgb",
    "num_cols": NUM_COLS,
    "cat_cols": CAT_COLS,
}

# Ridge pipeline
ridge_pipeline = {
    "preprocessor": preprocessor,
    "model_h1": ridge_h1,
    "model_h24": ridge_h24,
    "model_type": "ridge",
    "num_cols": NUM_COLS,
    "cat_cols": CAT_COLS,
}

# MLP pipeline
mlp_pipeline = {
    "preprocessor": preprocessor,
    "model_h1": mlp_h1,
    "model_h24": mlp_h24,
    "model_type": "mlp",
    "num_cols": NUM_COLS,
    "cat_cols": CAT_COLS,
}

print("Model pipelines packaged. Ready for final evaluation.")
print("\nUsage:")
print('  result = evaluate_final_model("path/to/test.csv", lgb_pipeline, horizon=1)')
print('  result = evaluate_final_model("path/to/test.csv", lgb_pipeline, horizon=24)')

In [ ]:
# --- Validate the evaluation function on training data (sanity check) ---
print("Sanity check: evaluating LightGBM pipeline on training file...\n")
sanity_result = evaluate_final_model(
    str(TRAIN_DATA_PATH),
    lgb_pipeline,
    horizon=1
)

## 8. Cross-Validation with TimeSeriesSplit (Optional Robustness Check)

In [ ]:
# --- TimeSeriesSplit CV for LightGBM (Horizon 1, raw target) ---
# Provides a more robust estimate than a single temporal split.

tscv = TimeSeriesSplit(n_splits=5)
cv_scores = []

X_full = preprocessor.fit_transform(df_clean)
y_full_h1 = df_clean["target_h1"].values

for fold, (train_idx, val_idx) in enumerate(tscv.split(X_full)):
    X_tr, X_vl = X_full[train_idx], X_full[val_idx]
    y_tr, y_vl = y_full_h1[train_idx], y_full_h1[val_idx]
    
    model_cv = lgb.LGBMRegressor(**lgb_params)
    model_cv.fit(X_tr, y_tr, eval_set=[(X_vl, y_vl)])
    
    preds_cv = model_cv.predict(X_vl)
    fold_mse = mean_squared_error(y_vl, preds_cv)
    cv_scores.append(fold_mse)
    print(f"  Fold {fold+1}: MSE = {fold_mse:.2f}")

print(f"\nTimeSeriesSplit CV — Mean MSE: {np.mean(cv_scores):.2f} ± {np.std(cv_scores):.2f}")

---
## Summary

This notebook implements a complete time-series forecasting pipeline for hourly bike counts:

1. **Feature Engineering:** Cyclical encodings, lag/rolling features, interaction terms, robust text parsing
2. **Models:** Ridge (linear), LightGBM (tree-based), MLP (neural network)
3. **Horizons:** t+1 single-step and t+1..t+24 multi-step
4. **Validation:** Strict temporal split + TimeSeriesSplit CV
5. **Production:** `evaluate_final_model()` for seamless inference on new data

**Next steps:** Hyperparameter tuning, ensemble methods, LSTM/Transformer architectures.